# London Tube Station Gravity Model Heatmap

This notebook visualizes the results of a gravity model for potential new Tube stations in London. It includes:

1. City-wide gravity score map.
2. Top 10 candidate locations map.
3. Summary table of top 10 locations.

## 1. Import Required Libraries

In [1]:
import os
import pandas as pd
import numpy as np
import folium
import branca.colormap as cm

## 2. Load the Data

In [2]:
# Set script directory
SCRIPT_DIR = os.path.dirname(os.path.abspath('__file__'))

# Load CSV
grid_path = r"C:\Users\Jonathan\OneDrive\Documents\SIG_Datathon\Output\sorted_grid.csv"
grid = pd.read_csv(grid_path)

print(f"Loaded {len(grid)} grid points")
print(f"gravity_log range: {grid['gravity_log'].min():.4f} – {grid['gravity_log'].max():.4f}")

Loaded 1954 grid points
gravity_log range: 0.0000 – 4.6101


## 3. City-Wide Gravity Score Map

Create a folium map showing gravity scores for all grid points across London.

In [14]:
center_lat = float(grid['lat'].mean())
center_lon = float(grid['lon'].mean())

# Initialize map
m_all = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles=None)
folium.TileLayer('OpenStreetMap', name='OpenStreetMap').add_to(m_all)
folium.TileLayer('CartoDB positron', name='CartoDB Positron').add_to(m_all)

# Define colormap
vmin, vmax = float(grid['gravity_log'].min()), float(grid['gravity_log'].max())
cmap = cm.LinearColormap(
    ['#08306b', '#2171b5', '#6baed6', '#ef6548', '#cb181d', '#99000d'],
    vmin=vmin, vmax=vmax
)
cmap.caption = 'Gravity Score (log)'

# Add grid points
for _, r in grid.iterrows():
    folium.CircleMarker(
        location=[float(r['lat']), float(r['lon'])],
        radius=5.0,
        color=cmap(float(r['gravity_log'])),
        fill=True,
        fill_opacity=0.8,
        weight=0,
        popup=(
            f"gravity_score={r['gravity_score']:.2f}<br>"
            f"gravity_log={r['gravity_log']:.4f}<br>"
            f"lat={r['lat']:.6f}, lon={r['lon']:.6f}"
        ),
    ).add_to(m_all)

cmap.add_to(m_all)
folium.LayerControl(collapsed=False).add_to(m_all)

# Save map
city_html = os.path.join(SCRIPT_DIR, 'gravity_citywise_score_map.html')
m_all.save(city_html)
print(f"Saved city-wise score map to: {city_html}")

Saved city-wise score map to: c:\Users\Jonathan\OneDrive\Documents\SIG_Datathon\gravity_citywise_score_map.html


## 4. Top 10 Candidate Locations Map

Highlight the top 10 locations from the gravity model with large red circles and numbered labels.

In [15]:
top10 = grid.sort_values('gravity_log', ascending=False).head(10).reset_index(drop=True)

m_top = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles=None)
folium.TileLayer('CartoDB positron', name='CartoDB Positron').add_to(m_top)
folium.TileLayer('OpenStreetMap', name='OpenStreetMap').add_to(m_top)

# Background grid in muted color
for _, r in grid.iterrows():
    folium.CircleMarker(
        location=[float(r['lat']), float(r['lon'])],
        radius=4.0,
        color=cmap(float(r['gravity_log'])),
        fill=True,
        fill_opacity=0.25,
        weight=0
    ).add_to(m_top)

# Highlight top 10
for rank, (_, row) in enumerate(top10.iterrows(), start=1):
    lat, lon = float(row['lat']), float(row['lon'])
    
    folium.CircleMarker(
        location=[lat, lon],
        radius=16,
        color='#d7191c',
        fill=True,
        fill_color='#d7191c',
        fill_opacity=0.85,
        weight=2,
        popup=folium.Popup(
            f"<b>Rank #{rank}</b><br>"
            f"gravity_score={row['gravity_score']:.2f}<br>"
            f"gravity_log={row['gravity_log']:.4f}<br>"
            f"lat={row['lat']:.6f}, lon={row['lon']:.6f}",
            max_width=300
        ),
    ).add_to(m_top)
    
    # Add rank label
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=(
                f'<div style="font-size:12px; font-weight:bold; color:white; '
                f'text-align:center; line-height:26px; width:26px; height:26px;">{rank}</div>'
            ),
            icon_size=(26,26),
            icon_anchor=(13,13)
        )
    ).add_to(m_top)

folium.LayerControl(collapsed=False).add_to(m_top)

top10_html = os.path.join(SCRIPT_DIR, 'gravity_top10_map.html')
m_top.save(top10_html)
print(f"Saved top-10 map: {top10_html}")


Saved top-10 map: c:\Users\Jonathan\OneDrive\Documents\SIG_Datathon\gravity_top10_map.html


## 5. Top 10 Gravity Model Summary Table

In [5]:
print("\nTop 10 Gravity Model Locations:")
for rank, (_, row) in enumerate(top10.iterrows(), start=1):
    print(f"  #{rank:>2d}  ({row['lat']:.4f}, {row['lon']:.4f})  "
          f"gravity_log={row['gravity_log']:.4f}  "
          f"gravity_score={row['gravity_score']:.2f}")


Top 10 Gravity Model Locations:
  # 1  (51.5145, -0.4125)  gravity_log=4.6101  gravity_score=99.49
  # 2  (51.3487, -0.1454)  gravity_log=3.4313  gravity_score=29.92
  # 3  (51.4594, -0.3281)  gravity_log=2.8690  gravity_score=16.62
  # 4  (51.4265, 0.0447)  gravity_log=2.8527  gravity_score=16.33
  # 5  (51.4876, 0.1483)  gravity_log=2.6464  gravity_score=13.10
  # 6  (51.4624, 0.0464)  gravity_log=2.6099  gravity_score=12.60
  # 7  (51.5368, -0.0945)  gravity_log=2.3901  gravity_score=9.91
  # 8  (51.5436, 0.0356)  gravity_log=2.3671  gravity_score=9.67
  # 9  (51.5256, 0.0348)  gravity_log=2.3641  gravity_score=9.63
  #10  (51.5885, 0.0376)  gravity_log=2.3390  gravity_score=9.37
